In [1]:
import torch

# char point to int. 
# char from unicode to unicode char point
# nothing fancy they are dedicated for unicode
print(ord('牛'))
print(chr(29275))

# Problem 1
# What Unicode character does chr(0) return?
# NULL char in C. but Py not using this to  respresent the end of the string.
print(chr(0))

print(repr(123))
print(repr('abc'))

# object can has both __repr__ and __str__ func impl
# representation, and human debugging form.
# repr: str typed representation of the obj.
c = chr(0)
r = repr(c)

print(r)           # visible: '\x00'
print(len(r))      # 6 characters: ', \, x, 0, 0, '

print(c)           # invisible
print(len(c))      # 1 character

# Problem 1.b & 1.c
# How does this character’s string representation (__repr__()) differ from its printed representation?
# Its repr() shows an escaped, visible form like '\x00', while its printed form outputs
# the actual null character, which has no visible representation.
# of course, chr(29275), str and repr form are the same.


29275
牛
 
123
'abc'
'\x00'
6
 
1


In [1]:
test_string = "hello! こんにちは!"

utf8_encoded = test_string.encode("utf-8")
# print the __str__ form, which is the same as __repr__ form in bytes class
# behavior: if ascii available, print; otherwise, escape with \xab
# why first part is hello? utf8 by design ascii compataible using the same asci mapping.
print(utf8_encoded) 
print(type(utf8_encoded))

# Get the byte values for the encoded string (integers from 0 to 255)
print(list(utf8_encoded))

# One byte does not necessarily correspond to one Unicode character!
# Note: py code itself utf8; plus the ascii comptabile so it's fine! smart!
# then internally save as utf8 code point; 
# len itself is metadata, so returned directly (here is not __len__ override, but rather Cpython runtime
# implements it)
print(len(test_string))
print(len(utf8_encoded))

print(utf8_encoded.decode("utf-8"))

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
<class 'bytes'>
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]
13
23
hello! こんにちは!


In [3]:
# What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than
# UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various
# input strings
# two things.
# - efficient encoding less redudancy, easier to train by model
# mainly because utf8 is space efficient. others having padding zero.
# - good presenetation for original str structure itself.
# preserving ascii. good for english.
# multi char also has same prefix, 0xxxx for ascii, 110xxxx for 2 byte, 1110 for 3 byte, 10xx for continuation byte.
# and so on. a nice distribution / pattern if we based on the byte prediction.

# b, Why incorrect? 
# def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
#     return "".join([bytes([b]).decode("utf-8") for b in bytestring])
# Answer: any str with continuation or 2 (or more) byte code point 

# c, https://chatgpt.com/g/g-p-69e3202dd82c81919760a902a41df409/c/69e430d4-2154-83ea-be0b-3e1b2fc40f04 
# E4 BD, subset of 你 which requrise 3 byte, and only one continuation byte. so ok.
print(123)

123


In [16]:
# BPE
# Byte level by unicode alone is not enough; often appearing word like `the` can use just one token
# Iterative process to count the most frequenet subsequence of token, assign new token.
# Then we get BPE

# Pre tokenization idea, getting the same word and counting multi times.
# Then later merge we look at the word freq map rather than scan entire corupuse again and again

# |<enoftext>| special token, inserting in doc, not split or merge, hardcoded logic.

# 1. download
# 2. exmaple code; multi processing lib.

# Side note, within one merge of token, we can't parallelize.
# Dict in python, not thread safe; unlike Go sync.Map
# GIL disallow any cpu intensiv work in parallel. you can effectively do only
# select server: waiting on IO and comptuation in single thread.

from io import BytesIO
from cs336_basics.pretokenization_example import find_chunk_boundaries

import importlib
importlib.reload(tok)

# toy = b"12345$123$ab$"
# f = BytesIO(toy)

# len(toy) == 13, so desired_num_chunks = 6 gives chunk_size = 13 // 6 = 2.
# boundaries = find_chunk_boundaries(f, desired_num_chunks=6, split_special_token=b"$")
# print(boundaries)

# for start, end in zip(boundaries[:-1], boundaries[1:]):
#     print(start, end, toy[start:end])

out = tok.get_chunks('../data/tiny-1000.txt')
# print(out[0][:100])
print(out[1][:100ls


<|endoftext|>
Once upon a time, there was a little flower named Bloom. Bloom lived in a big garden w


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'worker' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt